In [1]:
# Make the project root (the parent of 'notebooks/') importable
import sys, pathlib
cwd = pathlib.Path.cwd()
if cwd.name == "notebooks":
    proj_root = cwd.parent
else:
    proj_root = cwd  # if you launched from project root
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

In [2]:
from pathlib import Path
import geopandas as gpd

In [3]:
from src.ingestion.download_nyc_311 import NYC311Downloader
from src.ingestion.download_nta_geojson import NtaGeoJSONDownloader
from src.ingestion.download_nyc_pluto import NYCPlutoDownloader
from src.features.pluto_nta import aggregate_pluto_to_nta, save_pluto_nta_features
from src.aggregate.build_pluto_nta_features import load_pluto_data, resolve_input_path
from src.aggregate.aggregate_311_to_nta import aggregate_311_to_nta, save_aggregated_311

### Download NTA GeoJson Data

In [ ]:
dl = NtaGeoJSONDownloader(
    out_dir=Path("../data/raw/nyc/geographies")
)

In [ ]:
_ =  dl.download(filename="nyc_ntas_2020.geojson")

### Download Pluto Data

In [ ]:
downloader = NYCPlutoDownloader(
    query_file=Path("../src/queries/nyc_pluto.soql"),
    out_dir=Path("../data/raw/nyc/pluto"),
)


In [ ]:
downloader.run()

### 311 Data

In [ ]:
nyc_311_dl =  NYC311Downloader(
              query_file=Path("../src/queries/nyc_311.soql"),
              out_dir=Path("../data/raw/nyc/311")  
             )

In [ ]:
nyc_311_dl.run(start_date="2022-01-01", end_date="2025-12-31")

### Aggregate PLUTO into NTA

In [4]:
pluto_df = load_pluto_data(Path("../data/raw/nyc/pluto/nyc_pluto.csv"))


In [5]:
nta_path = resolve_input_path(Path("../data/raw/nyc/geographies/nyc_ntas_2020.geojson"), ("*.geojson", "*.gpkg", "*.shp"))

In [6]:
nta_gdf = gpd.read_file(nta_path)

In [7]:
features = aggregate_pluto_to_nta(pluto_df, nta_gdf)

In [8]:
saved_path = save_pluto_nta_features(features, Path("../data/processed/features/pluto_nta_features.parquet"))

### Aggregate 311 into NTA

In [ ]:
aggregated = aggregate_311_to_nta(
    Path("../data/raw/nyc/311/by_month"),
    Path("../data/raw/nyc/geographies/nyc_ntas_2020.geojson")
)

In [ ]:
_ = save_aggregated_311(aggregated, Path("../data/processed/features/complaints_311_nta_category.parquet"))